In [ ]:
import polars as pl
from settings import load_settings
from capstone.features.compute_repo_age_and_staleness import get_repo_age_and_staleness

In [ ]:
SETTINGS = load_settings()
OUTPUT_PATH = f'{SETTINGS.features_data_path}/feature_repo_age_and_staleness_retry.parquet'


In [ ]:
#init_df = pl.scan_parquet(SETTINGS.initial_dataset_path)
gh_df = pl.scan_parquet(SETTINGS.feature_repo_age_and_staleness_path)

In [ ]:
gh_df.collect_schema()

In [ ]:
# unique_repos = (
#     init_df
#         .select(
#             pl.col("package_name"),
#             pl.col("github_repo"),
#         )
#         .unique()
#         .collect()
# )

In [ ]:
gh_df.describe()

In [ ]:
nulls_df = (
    gh_df
        .filter(pl.col("repo_age_years").is_null() | pl.col("commit_staleness_days").is_null())
        .select(
            pl.col('package_name'),
            pl.col("github_repo"),
        
        )
        .collect()
)

In [ ]:
nulls_df.count()

In [ ]:
enriched_unique_repos = (
        nulls_df
            .with_columns(pl.col("github_repo")
            .map_elements(
                lambda repo: get_repo_age_and_staleness(repo),
                return_dtype=pl.Struct(
                    [
                        pl.Field("repo_age_years", pl.Float64),
                        pl.Field("commit_staleness_days", pl.Int64),
                    ]
                )
            )
            .alias("repo_stats")
        )
        .with_columns(
            pl.col("repo_stats").struct.field("repo_age_years"),
            pl.col("repo_stats").struct.field("commit_staleness_days"),
        )
        .drop("repo_stats")
    )

enriched_unique_repos.write_parquet(OUTPUT_PATH)

In [ ]:
enriched_unique_repos.describe()